# 01 · Dataset exploration

**Goal:** decide whether an image folder is fit for reconstruction before running feature extraction.

Checks: readable images, resolutions, blur, near-duplicates, and which intrinsics (`K`) each image will use at working resolution.

Set `SCENE_DIR` below (or the `MV3D_SCENE` environment variable) and run all cells.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.data.dataset import prepare_dataset
from src.data.validation import format_report, hamming

SCENE_DIR = Path(os.environ.get('MV3D_SCENE', '../data/raw/sample_scene'))
CONFIG_PATH = None  # None -> configs/default.yaml

config = load_config(CONFIG_PATH)
SCENE_DIR.resolve()

## 1. Load and validate

In [ ]:
dataset = prepare_dataset(SCENE_DIR, config)
report = dataset.report
print(format_report(report))

## 2. Thumbnails

Look for unrelated photos, motion blur, and gaps in coverage.

In [ ]:
n = len(dataset.images)
cols = min(6, n)
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.2 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis('off')
for ax, im in zip(axes.flat, dataset.images, strict=False):  # spare axes stay blank
    ax.imshow(im.image[..., ::-1])  # BGR -> RGB
    ax.set_title(im.name, fontsize=8)
fig.tight_layout()

## 3. Sharpness

Variance of the Laplacian at ≤ 800 px. Bars below the dashed threshold are flagged as blurry and are candidates for removal or recapture.

In [ ]:
blur = pd.Series({s.name: s.blur for s in report.images})
threshold = config.dataset.blur_threshold
colors = ['tab:red' if b < threshold else 'tab:blue' for b in blur]
fig, ax = plt.subplots(figsize=(max(6, 0.35 * len(blur)), 3))
ax.bar(range(len(blur)), blur.values, color=colors)
ax.axhline(threshold, color='k', ls='--', lw=1, label=f'threshold {threshold:g}')
ax.set_xticks(range(len(blur)), blur.index, rotation=90, fontsize=7)
ax.set_ylabel('Laplacian variance')
ax.set_yscale('log')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()

## 4. Similarity between images

dHash Hamming distance (0 = identical, ~32 = unrelated). Consecutive shots of a well-captured sequence show a bright band next to the diagonal; values ≤ `duplicate_max_distance` away from the diagonal are near-duplicates.

In [ ]:
hashes = [s.dhash for s in report.images]
names = [s.name for s in report.images]
D = np.array([[hamming(a, b) for b in hashes] for a in hashes])
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(D, cmap='viridis_r', vmin=0, vmax=32)
ax.set_xticks(range(len(names)), names, rotation=90, fontsize=6)
ax.set_yticks(range(len(names)), names, fontsize=6)
fig.colorbar(im, ax=ax, label='Hamming distance')
fig.tight_layout()

## 5. Intrinsics at working resolution

`source` is `file` (calibrated), `exif` (35 mm-equivalent focal length) or `fallback` (guess). Fallback intrinsics work for a first reconstruction but limit accuracy — calibrate if possible.

In [ ]:
pd.DataFrame(
    [
        {
            'name': im.name,
            'original': f'{im.original_size[0]}x{im.original_size[1]}',
            'working': f'{im.width}x{im.height}',
            'scale': round(im.scale[0], 4),
            'fx': round(im.K[0, 0], 1),
            'fy': round(im.K[1, 1], 1),
            'cx': round(im.K[0, 2], 1),
            'cy': round(im.K[1, 2], 1),
            'source': im.intrinsics_source,
        }
        for im in dataset.images
    ]
).set_index('name')

## 6. Findings

Record for this scene before moving on to feature extraction:

- Readable / skipped images:
- Blurry images to drop or recapture:
- Near-duplicates:
- Intrinsics source and whether calibration is needed:
- Coverage gaps seen in the thumbnails: